# FD-GAN Image Dehazing - Training

**Step 1:** Runtime -> Change runtime type -> **GPU (T4)**

**Step 2:** Upload ONLY the code files (left sidebar -> Files -> Upload):
- `model.py`
- `discriminator.py`
- `losses.py`
- `dataset.py`
- `train.py`
- `infer.py`

**Step 3:** Run all cells below in order.

In [ ]:
# === CELL 1: Check GPU ===
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# === CELL 2: Setup Kaggle API ===
# Go to kaggle.com -> Settings -> API -> Create New Token
# Copy the token string (starts with KGAT_...)

!pip install -q kaggle

import os

# PASTE YOUR KAGGLE TOKEN BELOW between the quotes
KAGGLE_TOKEN = ''  # <-- paste your KGAT_xxxx token here

if not KAGGLE_TOKEN:
    KAGGLE_TOKEN = input('Paste your Kaggle API token (KGAT_...): ')

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

# Save to file for CLI usage
!mkdir -p ~/.kaggle
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(KAGGLE_TOKEN)
!chmod 600 ~/.kaggle/access_token

print('Kaggle API configured!')

In [ ]:
# === CELL 3: Download RESIDE-6K dataset ===
!kaggle datasets download -d thedevastator/reside-6k -p /content/data_raw --unzip
print('Download complete!')

In [ ]:
# === CELL 4: Organize dataset ===
import os
from pathlib import Path

data_raw = Path('/content/data_raw')

print('Found folders:')
for p in sorted(data_raw.rglob('*')):
    if p.is_dir():
        imgs = list(p.glob('*.png')) + list(p.glob('*.jpg'))
        if len(imgs) > 0:
            print(f'  {p.relative_to(data_raw)}: {len(imgs)} images')

for d in ['data/train/hazy', 'data/train/clean', 'data/test/hazy', 'data/test/clean']:
    os.makedirs(d, exist_ok=True)

gt_names = ['GT', 'gt', 'clear', 'clean']

def find_gt(parent):
    for name in gt_names:
        if (parent / name).exists(): return parent / name
    return None

def setup_split(src, dst_hazy, dst_clean, name):
    hazy_src = src / 'hazy'
    clean_src = find_gt(src)
    if not hazy_src.exists() or clean_src is None:
        print(f'  {name}: not found'); return
    for f in hazy_src.glob('*.*'):
        dst = Path(dst_hazy) / f.name
        if not dst.exists(): os.symlink(f, dst)
    for f in clean_src.glob('*.*'):
        dst = Path(dst_clean) / f.name
        if not dst.exists(): os.symlink(f, dst)
    print(f'  {name}: {len(os.listdir(dst_hazy))} hazy, {len(os.listdir(dst_clean))} clean')

for root_candidate in [data_raw, data_raw / 'RESIDE-6K']:
    train_src = root_candidate / 'train'
    test_src = root_candidate / 'test'
    if train_src.exists() and (train_src / 'hazy').exists():
        setup_split(train_src, 'data/train/hazy', 'data/train/clean', 'Train')
        if test_src.exists():
            setup_split(test_src, 'data/test/hazy', 'data/test/clean', 'Test')
        break

print(f'\nReady!')

In [ ]:
# === CELL 5: Verify everything ===
from dataset import DehazingDataset
from model import ModernFDGAN
from discriminator import NLayerDiscriminator
from losses import VGGPerceptualLoss, GANLoss
from torch.utils.data import DataLoader
import torch.nn as nn

ds = DehazingDataset('data/train/hazy', 'data/train/clean', crop_size=256)
print(f'Dataset: {len(ds)} pairs')

gen = ModernFDGAN().to(device)
disc = NLayerDiscriminator(in_channels=6).to(device)
print(f'Generator:     {sum(p.numel() for p in gen.parameters()):,} params')
print(f'Discriminator: {sum(p.numel() for p in disc.parameters()):,} params')

x = torch.randn(2, 3, 256, 256).to(device)
with torch.no_grad(): y = gen(x)
print(f'Forward: {x.shape} -> {y.shape} OK!')

In [ ]:
# === CELL 6: TRAIN ===
import torch.optim as optim
import time

EPOCHS = 100
BATCH_SIZE = 8
CROP_SIZE = 256
LR = 2e-4
LAMBDA_L1 = 10.0
LAMBDA_PERC = 1.0
LAMBDA_GAN = 1.0

train_ds = DehazingDataset('data/train/hazy', 'data/train/clean', crop_size=CROP_SIZE, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

gen = ModernFDGAN().to(device)
disc = NLayerDiscriminator(in_channels=6, ndf=64, n_layers=3).to(device)

opt_g = optim.Adam(gen.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = optim.Adam(disc.parameters(), lr=LR, betas=(0.5, 0.999))

criterion_l1 = nn.L1Loss()
criterion_perc = VGGPerceptualLoss().to(device)
criterion_gan = GANLoss().to(device)

os.makedirs('checkpoints', exist_ok=True)

print(f'Training: {EPOCHS} epochs, {len(train_ds)} images, batch={BATCH_SIZE}')
print(f'Steps/epoch: {len(train_loader)}')
print('=' * 60)

best_loss = float('inf')
history = {'g': [], 'd': []}

for epoch in range(EPOCHS):
    t0 = time.perf_counter()
    gen.train(); disc.train()

    if epoch > EPOCHS // 2:
        decay = 1.0 - (epoch - EPOCHS // 2) / (EPOCHS // 2)
        for pg in opt_g.param_groups: pg['lr'] = LR * decay
        for pg in opt_d.param_groups: pg['lr'] = LR * decay

    ep_g, ep_d = 0.0, 0.0

    for i, batch in enumerate(train_loader):
        hazy = batch['hazy'].to(device)
        clean = batch['clean'].to(device)

        opt_d.zero_grad()
        with torch.no_grad(): fake = gen(hazy)
        ld = 0.5 * (criterion_gan(disc(hazy, clean), True) + criterion_gan(disc(hazy, fake), False))
        ld.backward(); opt_d.step()

        opt_g.zero_grad()
        fake = gen(hazy)
        lg = LAMBDA_GAN * criterion_gan(disc(hazy, fake), True) + LAMBDA_L1 * criterion_l1(fake, clean) + LAMBDA_PERC * criterion_perc(fake, clean)
        lg.backward(); opt_g.step()

        ep_g += lg.item(); ep_d += ld.item()

        if (i+1) % 200 == 0:
            print(f'  [{i+1}/{len(train_loader)}] G={lg.item():.4f} D={ld.item():.4f}')

    avg_g = ep_g / len(train_loader)
    avg_d = ep_d / len(train_loader)
    history['g'].append(avg_g); history['d'].append(avg_d)
    elapsed = time.perf_counter() - t0

    status = ''
    torch.save(gen.state_dict(), 'checkpoints/latest_gen.pth')
    if avg_g < best_loss:
        best_loss = avg_g
        torch.save(gen.state_dict(), 'checkpoints/best_gen.pth')
        status = ' *BEST*'
    if (epoch+1) % 10 == 0:
        torch.save(gen.state_dict(), f'checkpoints/gen_e{epoch+1:03d}.pth')

    print(f'E{epoch+1}/{EPOCHS} | G={avg_g:.4f} D={avg_d:.4f} | {elapsed:.0f}s{status}')

torch.save(gen.state_dict(), 'checkpoints/generator_final.pth')
print('\nDONE! Best: checkpoints/best_gen.pth')

In [ ]:
# === CELL 7: Training curves ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['g']); axes[0].set_title('Generator Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(history['d'], color='orange'); axes[1].set_title('Discriminator Loss'); axes[1].grid(alpha=0.3)
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
# === CELL 8: Visual comparison ===
import numpy as np

gen_test = ModernFDGAN().to(device)
gen_test.load_state_dict(torch.load('checkpoints/best_gen.pth', map_location=device))
gen_test.eval()

test_ds = DehazingDataset('data/test/hazy', 'data/test/clean', crop_size=None, augment=False)

def to_img(t):
    t = t.squeeze(0).cpu()
    return (((t + 1) / 2).clamp(0, 1).numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
for row in range(3):
    sample = test_ds[row * 100]
    hazy_t = sample['hazy'].unsqueeze(0).to(device)
    with torch.no_grad(): dehazed = gen_test(hazy_t)

    axes[row][0].imshow(to_img(sample['hazy'].unsqueeze(0))); axes[row][0].set_title('Hazy')
    axes[row][1].imshow(to_img(dehazed)); axes[row][1].set_title('Dehazed')
    axes[row][2].imshow(to_img(sample['clean'].unsqueeze(0))); axes[row][2].set_title('Ground Truth')
    for ax in axes[row]: ax.axis('off')

plt.tight_layout(); plt.savefig('comparison.png', dpi=150); plt.show()

In [ ]:
# === CELL 9: Download trained model ===
from google.colab import files
files.download('checkpoints/best_gen.pth')

# Then run locally:
# python infer.py --input hazy_photo.jpg --checkpoint best_gen.pth